# Module C0 — Architecture multi-fichiers

## Section 1 — Rappels synthétiques

### Ce qu'on a vu en Phase 2 (P09)
- `main.py` orchestre : il importe et appelle, sans contenir de logique
- Chaque fichier `.py` a une responsabilité unique
- `if __name__ == '__main__':` isole l'exécution de l'import

### Notion clé de ce module
| Terme | Définition |
|---|---|
| **Module** | Un fichier `.py` importable |
| **Import** | `from X import Y` : récupère Y depuis le module X |
| **Graphe de dépendances** | Schéma montrant qui importe qui |
| **Import circulaire** | A importe B ET B importe A → erreur Python |
| **Responsabilité unique** | Un fichier = une seule mission bien définie |


## Section 2 — Exercice guidé : comprendre le graphe de dépendances

### Étape 1 — Lire les imports du projet

Voici les déclarations d'import réelles de chaque fichier Drone Rescue.
Tu vas les analyser pour reconstruire le graphe de dépendances.

In [ ]:
# Imports réels de chaque fichier (reproduits inline pour Google Colab)
imports_projet = {
    'main.py':       ['logique', 'console', 'logger'],
    'console.py':    ['logique', 'affichage', 'config', 'logger'],
    'logique.py':    ['config'],
    'affichage.py':  ['config'],
    'logger.py':     [],  # uniquement stdlib (os, datetime)
    'config.py':     [],  # uniquement stdlib (json, os)
}
print('Données chargées :', list(imports_projet.keys()))

### Étape 2 — Qui est au sommet ? Qui est à la base ?

Complete la fonction ci-dessous pour calculer :
- `importeurs(fichier)` : la liste des fichiers qui importent `fichier`
- Le **niveau** d'un fichier = 0 si personne ne l'importe (base), plus élevé si tout le monde l'importe

In [ ]:
def importeurs(fichier, graphe):
    """Retourne la liste des fichiers qui importent 'fichier'."""
    # TODO : parcourir graphe, retourner les clés dont la valeur contient 'fichier'
    # (enlever l'extension .py pour la comparaison)
    resultat = []
    for source, cibles in graphe.items():
        pass  # TODO : compléter ici
    return resultat

# Vérification attendue : ['main.py', 'console.py'] importent logique
assert 'main.py' in importeurs('logique', imports_projet)
assert 'console.py' in importeurs('logique', imports_projet)
assert importeurs('config', imports_projet) != []
assert importeurs('main.py', imports_projet) == []  # personne n'importe main
print('Étape 2 : OK')

### Étape 3 — Identifier les fichiers feuilles (sans dépendances internes)

Un fichier **feuille** n'importe aucun autre fichier du projet (seulement la stdlib).
Ce sont les briques de base : tout le monde peut les utiliser sans risque de cycle.

In [ ]:
def fichiers_feuilles(graphe):
    """Retourne la liste des fichiers sans dépendance vers le projet."""
    # TODO : retourner les clés dont la liste d'imports est vide
    pass

feuilles = fichiers_feuilles(imports_projet)
assert 'config.py' in feuilles
assert 'logger.py' in feuilles
assert 'main.py' not in feuilles
print('Fichiers feuilles :', feuilles)
print('Étape 3 : OK')

### Étape 4 — Détecter un import circulaire

Un import circulaire survient quand A importe B ET B importe A (directement ou indirectement).
Complète la fonction pour détecter si deux fichiers s'importent mutuellement.

In [ ]:
def import_circulaire(a, b, graphe):
    """Retourne True si a importe b ET b importe a."""
    nom_a = a.replace('.py', '')
    nom_b = b.replace('.py', '')
    # TODO : vérifier que nom_b est dans graphe[a] ET nom_a est dans graphe[b]
    pass

# Cas sain : pas de cycle dans le projet réel
assert import_circulaire('logique.py', 'console.py', imports_projet) == False

# Simulation d'un graphe bugué
graphe_bogue = {
    'logique.py': ['config', 'console'],  # ERREUR : logique importe console
    'console.py': ['logique', 'affichage', 'config'],
    'config.py':  [],
    'affichage.py': ['config'],
}
assert import_circulaire('logique.py', 'console.py', graphe_bogue) == True
print('Étape 4 : OK — import circulaire détecté correctement')

### Étape 5 — Bilan : dessiner le graphe à la main

Dans ton cahier ou sur une feuille, dessine le graphe complet de Drone Rescue.
- Chaque fichier = un nœud (rectangle)
- Chaque import = une flèche de A vers B (`A → B` signifie 'A importe B')
- Vérifie qu'aucune flèche ne revient en arrière (pas de cycle)

Le graphe attendu :
```
main.py  ──────────────────────────────────────────────────┐
  │                                                         │
  ├──► logique.py ──► config.py                            │
  │                                                         ▼
  ├──► console.py ──► logique.py                       logger.py
  │         │      ──► affichage.py ──► config.py
  │         └──────► config.py
  └──► logger.py
```

In [ ]:
# Synthèse : afficher l'ordre topologique (du plus indépendant au plus dépendant)
# C'est l'ordre dans lequel Python charge les modules au démarrage.
ordre = ['config.py', 'logger.py', 'affichage.py', 'logique.py', 'console.py', 'main.py']
print('Ordre de chargement Python (du plus indépendant au plus dépendant) :')
for i, f in enumerate(ordre, 1):
    print(f'  {i}. {f}')